# Step 5 · Combine everything into the poll's list

**What this notebook does:** joins the tables from the earlier steps into one list of people for the website.

| input | made by | what it has |
|---|---|---|
| `repec_people.csv` | `01_repec_rankings.ipynb` | about 5,600 economists: name, institution, ranks, deceased or not |
| `repec_profiles.csv` | `02_repec_profiles.ipynb` | each economist's homepage and first affiliation |
| `kalshi_odds.csv` | `03_kalshi.ipynb` | Kalshi's names and their chance of winning |
| `nobel_laureates.csv` | `04_nobel_laureates.ipynb` | everyone who has won the prize, with the year |

**The rules**

1. Everyone from RePEc is on the list, with their homepage. If the rankings give no institution, the first
   affiliation on their profile is used.
2. Everyone on Kalshi's market gets their Kalshi chance. Anyone RePEc doesn't have is **added**.
3. Past winners are marked as **won** (with the year). They stay on the list so people can find them, but they
   can't be picked. Winners RePEc doesn't have are added too.
4. Economists RePEc marks as having died are marked as **deceased**. They can't be picked either.

**What it saves**

- `candidates.csv`: the full list as a table, easy to open in Excel.
- `../site/candidates.js`: the same list in the form the website reads.

Run it again after re-running any of the earlier steps.

## Setup

In [1]:
import json
import unicodedata

import pandas as pd

## 1. Read the four tables

In [2]:
people = pd.read_csv("repec_people.csv")
profiles = pd.read_csv("repec_profiles.csv")
kalshi = pd.read_csv("kalshi_odds.csv")
laureates = pd.read_csv("nobel_laureates.csv")

print(len(people), "economists from RePEc,", len(profiles), "of their profiles read,",
      len(kalshi), "names from Kalshi,", len(laureates), "past winners")

5644 economists from RePEc, 5644 of their profiles read, 9 names from Kalshi, 99 past winners


## 2. Add homepages, and fill in missing institutions

`merge` adds each economist's homepage and affiliation from step 2, matching on the RePEc ID. `how="left"` keeps
everyone from step 1, even anyone whose profile couldn't be read.

The rankings leave the institution empty for some economists, often because their time is split between several
places (Joseph Stiglitz, for example). For them, we use the first affiliation on their profile.

In [3]:
people = people.merge(profiles, on="repec_id", how="left")

# fillna(""): empty cells become empty text, so we don't have to treat them specially later.
people["institution"] = people["institution"].fillna("")
people["homepage"] = people["homepage"].fillna("")
people["affiliation"] = people["affiliation"].fillna("")

# No institution from the rankings: use the affiliation from the profile.
no_institution = people["institution"] == ""
people.loc[no_institution, "institution"] = people.loc[no_institution, "affiliation"]

print((people["homepage"] != "").sum(), "of", len(people), "economists have a homepage")
print(no_institution.sum(), "had no institution in the rankings;",
      (people.loc[no_institution, "institution"] != "").sum(), "of them filled in from their profile")

4344 of 5644 economists have a homepage
630 had no institution in the rankings; 269 of them filled in from their profile


## 3. Matching names across the sources

The same person is often written differently in different places:

| RePEc | Kalshi or Nobel Prize |
|---|---|
| Robert J. Barro | Robert Barro |
| Richard Blundell (Richard W. Blundell) | Richard Blundell |
| Jordi Galí | Jordi Gali |
| Robert E. Lucas, Jr. | Robert Lucas |

So we compare a **simplified name**: no accents, lower case, no middle names or initials, no "Jr.", only the first
and last word. All four examples above become `robert barro`, `richard blundell`, `jordi gali`, `robert lucas`.

This is the only function in these notebooks, because we need the same steps for all three sources.

In [4]:
def simple_name(name):
    # 1. Keep only the part before a bracket: "Richard Blundell (Richard W. Blundell)" -> "Richard Blundell"
    name = name.split("(")[0]

    # 2. Remove accents: "Galí" -> "Gali". (NFKD splits "í" into "i" plus an accent mark; encoding to ASCII drops the mark.)
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")

    # 3. Lower case, and treat dots and commas as spaces: "Robert E. Lucas, Jr." -> "robert e  lucas  jr "
    name = name.lower().replace(".", " ").replace(",", " ")

    # 4. Keep the words, but not suffixes like "jr" and not single letters (middle initials like "e")
    words = []
    for word in name.split():
        if word not in ["jr", "sr", "ii", "iii"] and len(word) > 1:
            words.append(word)

    # 5. First and last word: "robert lucas"
    if len(words) == 0:
        return ""
    return words[0] + " " + words[-1]


# Try it:
for example in ["Robert J. Barro", "Richard Blundell (Richard W. Blundell)", "Jordi Galí", "Robert E. Lucas, Jr."]:
    print(example, "->", simple_name(example))

Robert J. Barro -> robert barro
Richard Blundell (Richard W. Blundell) -> richard blundell
Jordi Galí -> jordi gali
Robert E. Lucas, Jr. -> robert lucas


## 4. Prepare the RePEc table

We add a few columns:
- `key`: the simplified name, for matching,
- `best_rank`: the better of the person's two RePEc ranks (used when two economists share a simplified name),
- `won_year` and `kalshi_pct`: filled in below,
- `source`: where the person came from.

In [5]:
people["key"] = people["name"].apply(simple_name)
people["best_rank"] = people[["rank_all_years", "rank_last_10_years"]].min(axis=1)
people["won_year"] = 0
people["kalshi_pct"] = None
people["source"] = "RePEc"

people[["name", "key", "institution", "best_rank"]].head()

,name,key,institution,best_rank
0,Andrei Shleifer,andrei shleifer,Harvard University,1.0
1,Daron Acemoglu,daron acemoglu,Massachusetts Institute of Technology (MIT),1.0
2,James J. Heckman,james heckman,University of Chicago,3.0
3,Joseph Stiglitz,joseph stiglitz,Columbia University,4.0
4,Jean Tirole,jean tirole,Toulouse School of Economics (TSE),5.0


## 5. Add Kalshi's odds

For each name on Kalshi, look for the same simplified name in RePEc.
- **Found:** give that person Kalshi's chance. If two RePEc economists share the simplified name, the
  better-ranked one is taken (a Nobel contender is almost always that one).
- **Not found:** add the person to the list.

The printout lets you check every match.

In [6]:
for _, row in kalshi.iterrows():
    key = simple_name(row["name"])
    matches = people[people["key"] == key]

    if len(matches) > 0:
        best = matches.sort_values("best_rank").index[0]    # the row number of the best-ranked match
        people.loc[best, "kalshi_pct"] = row["chance_pct"]
        print(row["name"], "->", people.loc[best, "name"], "(" + people.loc[best, "institution"] + ")")
    else:
        new_person = {"repec_id": "", "name": row["name"], "institution": "", "homepage": "", "profile_url": "",
                      "deceased": False, "key": key, "won_year": 0, "kalshi_pct": row["chance_pct"], "source": "Kalshi"}
        people = pd.concat([people, pd.DataFrame([new_person])], ignore_index=True)
        print(row["name"], "-> not in RePEc, added")

Ariel Pakes -> Ariel Pakes (Harvard University)
Richard Blundell -> Richard Blundell (University College London (UCL))
Susan Athey -> Susan Athey (Stanford University)
Robert Barro -> Robert J. Barro (Harvard University)
Partha Dasgupta -> not in RePEc, added
Ernst Fehr -> Ernst Fehr (Universität Zürich)
Andreu Mas-Colell -> Andreu Mas-Colell (Barcelona School of Economics (BSE))
Matthew Rabin -> Matthew Rabin (University of California-Berkeley)
Luigi Zingales -> Luigi Zingales (University of Chicago)


## 6. Mark past winners

Same idea: for each winner, look for the simplified name. Found: note the year. Not found: add them, so that
someone searching for them sees "won in …" instead of "not on the list".

In [7]:
added = []

for _, row in laureates.iterrows():
    key = simple_name(row["name"])
    matches = people[people["key"] == key]

    if len(matches) > 0:
        best = matches.sort_values("best_rank").index[0]
        people.loc[best, "won_year"] = row["year"]
    else:
        new_person = {"repec_id": "", "name": row["name"], "institution": "", "homepage": "", "profile_url": "",
                      "deceased": False, "key": key, "won_year": row["year"], "kalshi_pct": None, "source": "Nobel Prize"}
        people = pd.concat([people, pd.DataFrame([new_person])], ignore_index=True)
        added.append(row["name"])

print((people["won_year"] > 0).sum(), "past winners marked;", len(added), "of them were not in RePEc and were added:")
print(", ".join(added))

99 past winners marked; 17 of them were not in RePEc and were added:
Leonid Hurwicz, Thomas C. Schelling, Robert Mundell, Trygve Haavelmo, Maurice Allais, Richard Stone, Gerard Debreu, Sir Arthur Lewis, Bertil Ohlin, James E. Meade, Leonid Vitaliyevich Kantorovich, Tjalling C. Koopmans, Friedrich von Hayek, Gunnar Myrdal, Wassily Leontief, Jan Tinbergen, Ragnar Frisch


Check the matched winners: each RePEc name and institution should be the actual laureate, not someone with
the same name.

In [8]:
people[(people["won_year"] > 0) & (people["source"] == "RePEc")][["won_year", "name", "institution"]].sort_values("won_year")

,won_year,name,institution
146,1970,Paul A. Samuelson,
1407,1971,Simon Kuznets,
230,1972,Kenneth J. Arrow,
3017,1972,John R. Hicks,
294,1976,Milton Friedman,
...,...,...,...
173,2024,Simon Johnson,National Bureau of Economic Research (NBER)
1,2024,Daron Acemoglu,Massachusetts Institute of Technology (MIT)
207,2025,Peter Howitt,Brown University
14,2025,Philippe Aghion,London School of Economics (LSE)


## 7. Status and ID

- `status`: **won** if they won the prize, otherwise **deceased** if RePEc marks them so, otherwise **eligible**.
- `id`: a short name the website uses for each person: the RePEc ID (like `pat6`), or, for people added from
  Kalshi or the Nobel list, their simplified name with dashes (like `partha-dasgupta`).

In [9]:
people["status"] = "eligible"
people.loc[people["deceased"] == True, "status"] = "deceased"
people.loc[people["won_year"] > 0, "status"] = "won"          # winning is the main reason, so it is set last

people["id"] = people["repec_id"].fillna("")
no_id = people["id"] == ""
people.loc[no_id, "id"] = people.loc[no_id, "key"].str.replace(" ", "-")

people["status"].value_counts()

status
eligible    5311
deceased     252
won           99
Name: count, dtype: int64

## 8. Save the table

Sorted so Kalshi's names come first (highest chance first), then everyone else by RePEc rank.

In [10]:
people = people.sort_values(["kalshi_pct", "best_rank"], ascending=[False, True], na_position="last")

columns = ["id", "name", "institution", "homepage", "profile_url", "status", "won_year", "kalshi_pct",
           "rank_all_years", "rank_last_10_years", "source"]
people[columns].to_csv("candidates.csv", index=False)

print("Saved", len(people), "people to candidates.csv")
people[columns].head(12)

Saved 5662 people to candidates.csv


,id,name,institution,homepage,profile_url,status,won_year,kalshi_pct,rank_all_years,rank_last_10_years,source
136,ppa20,Ariel Pakes,Harvard University,https://scholar.harvard.edu/pakes,https://ideas.repec.org/e/ppa20.html,eligible,0,23,137.0,988.0,RePEc
187,pat6,Susan Athey,Stanford University,https://people.stanford.edu/athey/,https://ideas.repec.org/e/pat6.html,eligible,0,13,188.0,11.0,RePEc
17,pbl81,Richard Blundell,University College London (UCL),http://www.homepages.ucl.ac.uk/~uctp39a/,https://ideas.repec.org/e/pbl81.html,eligible,0,13,18.0,339.0,RePEc
5,pba251,Robert J. Barro,Harvard University,https://scholar.harvard.edu/barro,https://ideas.repec.org/f/pba251.html,eligible,0,11,6.0,54.0,RePEc
79,pfe29,Ernst Fehr,Universität Zürich,https://www.econ.uzh.ch/en/people/faculty/fehr...,https://ideas.repec.org/e/pfe29.html,eligible,0,10,80.0,413.0,RePEc
859,pma1422,Andreu Mas-Colell,Barcelona School of Economics (BSE),https://www.upf.edu/web/andreu-mas-colell/,https://ideas.repec.org/f/pma1422.html,eligible,0,10,860.0,NaN,RePEc
5644,partha-dasgupta,Partha Dasgupta,,,,eligible,0,10,NaN,NaN,Kalshi
254,pra660,Matthew Rabin,University of California-Berkeley,http://emlab.berkeley.edu/~rabin/,https://ideas.repec.org/f/pra660.html,eligible,0,9,255.0,NaN,RePEc
62,pzi101,Luigi Zingales,University of Chicago,http://faculty.chicagobooth.edu/luigi.zingales...,https://ideas.repec.org/e/pzi101.html,eligible,0,5,63.0,844.0,RePEc
0,psh93,Andrei Shleifer,Harvard University,https://scholar.harvard.edu/shleifer/home,https://ideas.repec.org/e/psh93.html,eligible,0,None,1.0,79.0,RePEc


## 9. Save the list for the website

The website loads `../site/candidates.js`. To keep that file small, each person is written as a short list in
this order:

`[id, name, institution, homepage, RePEc profile, status, year won, Kalshi %, best RePEc rank]`

The RePEc profile is shortened from `https://ideas.repec.org/e/pat6.html` to `e/pat6`; the website adds the
rest back. Empty values are written as `""` or `null`.

The file also says when Kalshi's odds were downloaded (step 3), so the page can say "Updated 40 minutes ago" under
the table. The time is written in the standard form browsers read: `2026-09-26T23:40:00Z` (Z means UTC).

In [11]:
rows = []

for _, p in people.iterrows():
    profile = ""
    if p["profile_url"]:
        profile = p["profile_url"].replace("https://ideas.repec.org/", "").replace(".html", "")   # "e/pat6"

    kalshi_pct = None
    if pd.notna(p["kalshi_pct"]):
        kalshi_pct = int(p["kalshi_pct"])

    best_rank = None
    if pd.notna(p["best_rank"]):
        best_rank = int(p["best_rank"])

    rows.append([p["id"], p["name"], p["institution"], p["homepage"], profile,
                 p["status"], int(p["won_year"]), kalshi_pct, best_rank])

# When Kalshi's odds were downloaded in step 3, e.g. "2026-09-26 23:40 UTC".
# Rewritten in the standard form browsers read: "2026-09-26T23:40:00Z".
kalshi_time = kalshi["downloaded_at"].iloc[0]
kalshi_time = kalshi_time.replace(" UTC", ":00Z").replace(" ", "T")

text = ("// Made by data/05_combine.ipynb. Don't edit by hand: re-run the notebooks instead.\n"
        "// Each person: [id, name, institution, homepage, RePEc profile, status, year won, Kalshi %, best RePEc rank]\n"
        "window.CANDIDATES = " + json.dumps(rows, ensure_ascii=False, separators=(",", ":")) + ";\n"
        "// When Kalshi's odds were downloaded (step 3)\n"
        'window.KALSHI_UPDATED = "' + kalshi_time + '";\n')

with open("../site/candidates.js", "w", encoding="utf-8") as f:
    f.write(text)

print("Saved", len(rows), "people to ../site/candidates.js (" + str(round(len(text.encode("utf-8")) / 1024)) + " KB)")
print("Kalshi's odds were downloaded at", kalshi_time)

Saved 5662 people to ../site/candidates.js (699 KB)
Kalshi's odds were downloaded at 2026-09-26T23:40:00Z
